# System Settings

In [26]:
import asyncio
import sys
import random
import json
import re
import math
import pandas as pd

from pathlib import Path
from datetime import datetime, timedelta
from urllib.parse import quote_plus
from playwright.async_api import async_playwright
import nest_asyncio
import pprint

pp = pprint.PrettyPrinter(indent=2, sort_dicts=False)

In [ ]:
# --- Environment Setup ---

# สำหรับ Jupyter Notebook
nest_asyncio.apply()

# สำหรับ Windows
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

print("✅ [Jupyter] Nested Event Loop: Enabled.")

✅ [Jupyter] Nested Event Loop: Enabled.


In [ ]:
# --- Config ---

USER_DATA_DIR = "./my_session"
OUTPUT_DIR = "./collected_data"         # Final output
TEMP_DIR = "./collected_data/temp"      # Temp checkpoint

# Log folders
LOG_DIR = "./log"
PRODUCT_LOG_DIR = "./log/product"
DETAIL_LOG_DIR = "./log/detail"
REVIEW_LOG_DIR = "./log/review"
SEARCH_LOG_DIR = "./log/search"

# Delay request
REQUEST_DELAY_MIN = 2.0
REQUEST_DELAY_MAX = 3.0

# เปลี่ยนเป็น True เฉพาะตอนต้องการ Login / สร้าง session ใหม่
CREATE_SESSION = False

# --- Checkpoint Frequency ---
PRODUCT_CHECKPOINT_EVERY_PAGES = 10
DETAIL_CHECKPOINT_EVERY_URLS = 10
REVIEW_CHECKPOINT_EVERY_PAGES = 50
SEARCH_CHECKPOINT_EVERY_PAGES = 50

# --- Resume Setting ---
RESUME_FROM_BACKUP = True

# สร้าง folder ถ้ายังไม่มี
for folder in [
    USER_DATA_DIR,
    OUTPUT_DIR,
    TEMP_DIR,
    LOG_DIR,
    PRODUCT_LOG_DIR,
    DETAIL_LOG_DIR,
    REVIEW_LOG_DIR,
    SEARCH_LOG_DIR,
]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print(f"🌐 Browser session folder: {USER_DATA_DIR}")
print(f"📂 Output folder: {OUTPUT_DIR}")
print(f"💾 Temp checkpoint folder: {TEMP_DIR}")
print(f"🧾 Log folder: {LOG_DIR}")

🌐 Browser session folder: ./my_session
📂 Output folder: ./collected_data
💾 Temp checkpoint folder: ./collected_data/temp
🧾 Log folder: ./log


In [ ]:
# --- Dedupe Columns ---

PRODUCT_DEDUPE_COLS = [
    "product_id",
    "sku_id",
    "product_url"
]

DETAIL_DEDUPE_COLS = [
    "product_id",
    "product_url"
]

REVIEW_DEDUPE_COLS = [
    "product_id",
    "user_name",
    "review_date_raw",
    "comment_text"
]

SEARCH_DEDUPE_COLS = [
    "search_keyword",
    "product_id",
    "sku_id",
    "product_url"
]

# Functions

## Browser Helpers

In [ ]:
async def create_browser_context(playwright, block_images=False): 
    """
    เปิด Chromium แบบ persistent context เพื่อเก็บ session/cookies

    หมายเหตุ:
    - ห้ามเปิดหลาย browser พร้อมกันโดยใช้ USER_DATA_DIR เดียวกัน
    """

    context = await playwright.chromium.launch_persistent_context(
        user_data_dir=USER_DATA_DIR,
        headless=False,
        viewport={"width": 1366, "height": 768},
        locale="th-TH",
        timezone_id="Asia/Bangkok",
    )

    page = context.pages[0] if context.pages else await context.new_page()

    if block_images:
        async def block_image_route(route):
            await route.abort()

        await page.route("**/*.{png,jpg,jpeg,webp,gif,svg}", block_image_route)

    return context, page


async def human_delay(
    min_delay=REQUEST_DELAY_MIN,
    max_delay=REQUEST_DELAY_MAX
):
    """
    หน่วงเวลาแบบสุ่ม เพื่อไม่ให้ request ถี่เกินไป
    """

    await asyncio.sleep(random.uniform(min_delay, max_delay))


def make_timestamp():
    """
    ใช้สร้าง timestamp สำหรับชื่อไฟล์
    """

    return datetime.now().strftime("%Y%m%d_%H%M%S")


def make_collected_at():
    """
    ใช้บันทึกเวลาที่ดึงข้อมูล
    """

    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


## Manual Lazada Session Setup

In [31]:
async def create_lazada_session():
    """
    ใช้สำหรับ Login และบันทึก Session ของ Browser

    ควรรันเฉพาะกรณี:
    - ใช้งานครั้งแรก
    - Session หมดอายุ
    - Lazada บังคับ Login ใหม่
    """

    async with async_playwright() as p:
        context, page = await create_browser_context(
            p,
            block_images=False
        )

        await page.goto(
            "https://www.lazada.co.th",
            wait_until="domcontentloaded",
            timeout=60000
        )

        print("\n" + "—" * 60)
        print("⚡ [SESSION INITIALIZED] Browser instance is active.")
        print("—" * 60)
        print("📋 ACTION REQUIRED:")
        print(" 1. Login Lazada manually.")
        print(" 2. Confirm default logistics/payment settings if required.")
        print(" 3. Close browser window when finished.")
        print("—" * 60 + "\n")

        while len(context.pages) > 0:
            await asyncio.sleep(1)

        print(f"✅ [SUCCESS] Session persisted successfully in '{USER_DATA_DIR}'.")


if CREATE_SESSION:
    await create_lazada_session()

## General Helpers

In [32]:
def safe_float(val, default=0.0):
    """
    แปลงค่าเป็น float
    """

    if val is None:
        return default

    text = str(val).strip()

    if text == "":
        return default

    try:
        text = (
            text.replace(",", "")
                .replace("฿", "")
                .replace("THB", "")
                .replace("thb", "")
                .strip()
        )

        return float(text)

    except Exception:
        return default


def safe_int(val, default=0):
    """
    แปลงค่าเป็น int
    """

    if val is None:
        return default

    text = str(val).strip()

    if text == "":
        return default

    try:
        text = (
            text.replace(",", "")
                .replace("฿", "")
                .replace("THB", "")
                .replace("thb", "")
                .strip()
        )

        return int(float(text))

    except Exception:
        return default


def parse_sold_count(sold_str):
    """
    แปลงยอดขายจากข้อความ Lazada เป็นตัวเลข

    ตัวอย่าง:
    - "9.4K ชิ้น" -> 9400
    - "1.2k sold" -> 1200
    - "10,000 ชิ้น" -> 10000
    - "2M sold" -> 2000000
    """

    if not sold_str:
        return 0

    s = (
        str(sold_str)
        .lower()
        .replace("ชิ้น", "")
        .replace("ขายแล้ว", "")
        .replace("sold", "")
        .replace(",", "")
        .strip()
    )

    if "k" in s:
        num = safe_float(s.replace("k", ""))
        return int(num * 1000)

    if "m" in s:
        num = safe_float(s.replace("m", ""))
        return int(num * 1_000_000)

    digits = re.sub(r"[^\d]", "", s)

    return safe_int(digits)


def clean_brand(brand):
    """
    ทำความสะอาดชื่อแบรนด์
    """

    if not brand:
        return ""

    brand = str(brand).strip()

    if re.search(r"no\s*brand", brand, re.IGNORECASE):
        return ""

    return brand


def clean_url(url):
    """
    แปลง URL ให้เป็น absolute URL
    """

    if not url:
        return ""

    url = str(url).strip()

    if url.startswith("http"):
        return url

    if url.startswith("//"):
        return f"https:{url}"

    if url.startswith("/"):
        return f"https://www.lazada.co.th{url}"

    return f"https://www.lazada.co.th/{url}"


def clean_full_text(text):
    """
    สำหรับ Description:
    - ยุบบรรทัดว่างซ้ำ
    - ยุบช่องว่างซ้ำ
    - ยังคงโครงสร้างหลายบรรทัดไว้
    """

    if not text:
        return ""

    text = str(text)
    text = re.sub(r"\n\s*\n", "\n", text)
    text = re.sub(r" +", " ", text)

    return text.strip()


def clean_minimal(text):
    """
    สำหรับ Specs / Qualification:
    ยุบทุกอย่างให้เหลือบรรทัดเดียว
    """

    if not text:
        return ""

    return re.sub(r"\s+", " ", str(text)).strip()


def safe_json_loads(raw_text, default=None):
    """
    แปลง JSON แบบปลอดภัย
    """

    if default is None:
        default = {}

    if not raw_text:
        return default

    try:
        return json.loads(raw_text)

    except Exception:
        return default

## Date Helper

In [33]:
THAI_ABBR_MONTHS = {
    "ม.ค.": "Jan",
    "ก.พ.": "Feb",
    "มี.ค.": "Mar",
    "เม.ย.": "Apr",
    "พ.ค.": "May",
    "มิ.ย.": "Jun",
    "ก.ค.": "Jul",
    "ส.ค.": "Aug",
    "ก.ย.": "Sep",
    "ต.ค.": "Oct",
    "พ.ย.": "Nov",
    "ธ.ค.": "Dec",
}


def thai_date_to_datetime(date_str):
    """
    แปลงวันที่เป็น YYYY-MM-DD

    รองรับ:
    - 05 Apr 2025
    - 05 ม.ค. 2025
    """

    if not date_str:
        return None

    translated = str(date_str).strip()

    for thai_abbr, eng_abbr in THAI_ABBR_MONTHS.items():
        if thai_abbr in translated:
            translated = translated.replace(thai_abbr, eng_abbr)
            break

    try:
        dt_obj = datetime.strptime(translated, "%d %b %Y")
        return dt_obj.strftime("%Y-%m-%d")

    except ValueError:
        return None


def convert_time_interval(interval):
    """
    แปลงเวลาจาก Lazada ให้เป็น YYYY-MM-DD
    """

    if not interval:
        return None

    reference_date = datetime.now()
    interval = str(interval).strip()
    interval_lower = interval.lower()

    if interval in ["วันนี้", "today", "Today"] or interval_lower == "just now":
        return reference_date.strftime("%Y-%m-%d")

    if interval in ["เมื่อวาน", "yesterday", "Yesterday"]:
        return (reference_date - timedelta(days=1)).strftime("%Y-%m-%d")

    parts = interval.split()

    if len(parts) >= 2:
        quantity = safe_int(parts[0], default=None)

        if quantity is not None:
            if any(key in interval for key in ["นาทีที่แล้ว", "minute ago", "minutes ago"]):
                dt = reference_date - timedelta(minutes=quantity)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["ชั่วโมงที่แล้ว", "hour ago", "hours ago"]):
                dt = reference_date - timedelta(hours=quantity)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["วันที่แล้ว", "day ago", "days ago"]):
                dt = reference_date - timedelta(days=quantity)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["สัปดาห์ที่แล้ว", "week ago", "weeks ago"]):
                dt = reference_date - timedelta(weeks=quantity)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["เดือนที่แล้ว", "month ago", "months ago"]):
                dt = reference_date - timedelta(days=quantity * 30)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["ปีที่แล้ว", "year ago", "years ago"]):
                dt = reference_date - timedelta(days=quantity * 365)
                return dt.strftime("%Y-%m-%d")

    formatted_date = thai_date_to_datetime(interval)

    if formatted_date:
        return formatted_date

    return interval

## Log / Temp Helpers

In [34]:
def get_scraper_log_paths(scraper_name):
    """
    คืน path ของ log file และ error jsonl ตามประเภท scraper
    """

    if scraper_name in ["product_by_shop", "product"]:
        return {
            "log_file": Path(PRODUCT_LOG_DIR) / "product_scraper.log",
            "error_file": Path(PRODUCT_LOG_DIR) / "product_error.jsonl",
        }

    if scraper_name in ["product_detail", "detail"]:
        return {
            "log_file": Path(DETAIL_LOG_DIR) / "product_detail_scraper.log",
            "error_file": Path(DETAIL_LOG_DIR) / "product_detail_error.jsonl",
        }

    if scraper_name in ["reviews", "review"]:
        return {
            "log_file": Path(REVIEW_LOG_DIR) / "review_scraper.log",
            "error_file": Path(REVIEW_LOG_DIR) / "review_error.jsonl",
        }

    if scraper_name in ["search_keyword", "search"]:
        return {
            "log_file": Path(SEARCH_LOG_DIR) / "search_keyword_scraper.log",
            "error_file": Path(SEARCH_LOG_DIR) / "search_keyword_error.jsonl",
        }

    return {
        "log_file": Path(LOG_DIR) / f"{scraper_name}.log",
        "error_file": Path(LOG_DIR) / f"{scraper_name}_error.jsonl",
    }


def append_text_log(scraper_name, level, message, context=None):
    """
    เขียน log แบบ text append ทีละบรรทัด
    """

    paths = get_scraper_log_paths(scraper_name)
    log_file = paths["log_file"]
    log_file.parent.mkdir(parents=True, exist_ok=True)

    context = context or {}

    log_line = (
        f"{make_collected_at()} | "
        f"{level.upper()} | "
        f"{scraper_name} | "
        f"{message} | "
        f"{json.dumps(context, ensure_ascii=False, default=str)}\n"
    )

    with open(log_file, "a", encoding="utf-8") as f:
        f.write(log_line)


def append_error_jsonl(scraper_name, record):
    """
    เขียน error/warning แบบ JSONL append ทีละ record
    """

    paths = get_scraper_log_paths(scraper_name)
    error_file = paths["error_file"]
    error_file.parent.mkdir(parents=True, exist_ok=True)

    with open(error_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def add_scrape_log(
    log_rows,
    scraper_name,
    level,
    message,
    **context
):
    """
    เพิ่ม log record ระหว่าง scraping

    - print บนหน้าจอ
    - append ลง .log
    - ถ้า WARNING/ERROR จะ append ลง .jsonl ด้วย
    """

    level = level.upper()

    log_record = {
        "logged_at": make_collected_at(),
        "scraper_name": scraper_name,
        "level": level,
        "message": message,
    }

    log_record.update(context)
    log_rows.append(log_record)

    if level == "ERROR":
        print(f"❌ [{scraper_name}] {message}")
        if context:
            print(f"   > Context: {context}")

    elif level == "WARNING":
        print(f"⚠️ [{scraper_name}] {message}")
        if context:
            print(f"   > Context: {context}")

    else:
        print(f"   ℹ️ [{scraper_name}] {message}")

    append_text_log(
        scraper_name=scraper_name,
        level=level,
        message=message,
        context=context
    )

    if level in ["WARNING", "ERROR"]:
        append_error_jsonl(scraper_name, log_record)

    return log_record


def save_scrape_checkpoint(
    rows,
    data_prefix,
    dedupe_cols=None,
    reason="checkpoint",
    overwrite_latest=True,
):
    """
    บันทึก checkpoint ระหว่าง scraping

    Temp data:
    - save เป็น CSV ลง collected_data/temp/
    """

    Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

    saved_files = {}

    if rows:
        df_data = pd.DataFrame(rows)

        if dedupe_cols:
            valid_dedupe_cols = [
                col for col in dedupe_cols
                if col in df_data.columns
            ]

            if valid_dedupe_cols:
                df_data = df_data.drop_duplicates(subset=valid_dedupe_cols)

        if overwrite_latest:
            data_file = Path(TEMP_DIR) / f"{data_prefix}.csv"
        else:
            data_file = Path(TEMP_DIR) / f"{data_prefix}_{make_timestamp()}.csv"

        df_data.to_csv(
            data_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"   💾 Temp checkpoint saved: {data_file} "
            f"| rows={len(df_data):,} "
            f"| reason={reason}"
        )

        saved_files["data"] = data_file

    else:
        print(f"⚠️ No collected data to save at checkpoint: {reason}")

    return saved_files


def save_final_data(
    rows,
    final_prefix,
    dedupe_cols=None
):
    """
    บันทึก final file เป็น Excel ลง collected_data/
    """

    if not rows:
        print(f"⚠️ No data to save for final file: {final_prefix}")
        return pd.DataFrame(), None

    df_final = pd.DataFrame(rows)

    if dedupe_cols:
        valid_dedupe_cols = [
            col for col in dedupe_cols
            if col in df_final.columns
        ]

        if valid_dedupe_cols:
            df_final = df_final.drop_duplicates(subset=valid_dedupe_cols)

    filename = Path(OUTPUT_DIR) / f"{final_prefix}_{make_timestamp()}.xlsx"
    df_final.to_excel(filename, index=False)

    print("\n" + "=" * 60)
    print(f"✅ FINAL FILE SAVED: {final_prefix}")
    print(f"📊 Total Rows: {len(df_final):,}")
    print(f"💾 File: {filename}")
    print("=" * 60)

    return df_final, filename


def save_failed_log(failed_rows, prefix, scraper_name="general"):
    """
    บันทึก failed log เป็น JSONL
    """

    if not failed_rows:
        return None

    paths = get_scraper_log_paths(scraper_name)
    error_file = paths["error_file"]
    error_file.parent.mkdir(parents=True, exist_ok=True)

    for row in failed_rows:
        record = {
            "logged_at": make_collected_at(),
            "type": prefix,
            **row
        }

        with open(error_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

    print(f"⚠️ Failed log appended: {error_file}")

    return error_file


def save_debug_html(html, prefix, key="unknown"):
    """
    บันทึก HTML สำหรับ debug กรณีดึง Product Detail ไม่ได้
    """

    Path(DETAIL_LOG_DIR).mkdir(parents=True, exist_ok=True)

    debug_file = Path(DETAIL_LOG_DIR) / f"{prefix}_{key}_{make_timestamp()}.html"
    debug_file.write_text(html, encoding="utf-8")

    print(f"⚠️ Debug HTML saved: {debug_file}")

    return debug_file


def load_existing_backup(data_prefix, dedupe_cols=None):
    """
    โหลด temp backup เดิมจาก collected_data/temp/

    รองรับ:
    - .csv เป็นหลัก
    - .xlsx เป็น fallback สำหรับไฟล์เก่า
    """

    csv_file = Path(TEMP_DIR) / f"{data_prefix}.csv"
    xlsx_file = Path(TEMP_DIR) / f"{data_prefix}.xlsx"

    if csv_file.exists():
        backup_file = csv_file
        read_func = pd.read_csv

    elif xlsx_file.exists():
        backup_file = xlsx_file
        read_func = pd.read_excel

    else:
        print(f"ℹ️ No existing temp backup found: {csv_file} or {xlsx_file}")
        return []

    try:
        df_existing = read_func(backup_file)

        if dedupe_cols:
            valid_dedupe_cols = [
                col for col in dedupe_cols
                if col in df_existing.columns
            ]

            if valid_dedupe_cols:
                df_existing = df_existing.drop_duplicates(subset=valid_dedupe_cols)

        rows = df_existing.to_dict(orient="records")

        print(f"🔁 Existing temp backup loaded: {backup_file} | rows={len(rows):,}")

        return rows

    except Exception as e:
        print(f"⚠️ Failed to load existing temp backup: {backup_file} | error={e}")
        return []


## Page Check Helpers

In [35]:
async def handle_page_check(page, label, max_wait_seconds=120):
    """
    ตรวจสอบว่าหน้าเป็น:
    - JSON API ปกติ
    - HTML ปกติ
    - หน้า verification

    หมายเหตุ:
    - ฟังก์ชันนี้ไม่ได้ bypass verification
    - ถ้าเจอ verification จะให้ผู้ใช้จัดการเองใน browser
    """

    start_time = datetime.now()

    while True:
        elapsed = (datetime.now() - start_time).total_seconds()

        if elapsed > max_wait_seconds:
            raise TimeoutError(f"Page check timeout at: {label}")

        try:
            content = (await page.inner_text("body", timeout=5000)).strip()

        except Exception:
            content = ""

        url_lower = page.url.lower()

        if not content:
            await asyncio.sleep(1)
            continue

        if content.startswith("{") or content.startswith("["):
            return content

        has_slider = False

        try:
            has_slider = await page.locator(
                "#nocaptcha, #px-captcha, .btn_slide"
            ).count() > 0

        except Exception:
            has_slider = False

        is_verify_url = (
            "verify" in url_lower
            or "punish" in url_lower
            or "captcha" in url_lower
        )

        is_verify_text = any(
            key.lower() in content.lower()
            for key in [
                "verify",
                "verification",
                "captcha",
                "security check",
                "กรุณายืนยัน",
                "ยืนยันตัวตน",
            ]
        )

        if not has_slider and not is_verify_url and not is_verify_text:
            return content

        prompt_msg = (
            f"\n >>>⚠️ [MANUAL CHECK REQUIRED] at {label}\n"
            f" >>> Please complete verification in the browser, then press ENTER to continue..."
        )

        await asyncio.get_event_loop().run_in_executor(None, input, prompt_msg)

        await page.wait_for_load_state("domcontentloaded")
        await asyncio.sleep(1)


# ------------------------------------------------------------
# Product Info Helpers
# ------------------------------------------------------------

def extract_product_id_from_url(url):
    """
    ดึง product_id จาก URL Lazada
    """

    if not url:
        return "unknown"

    url = str(url)

    match = re.search(r"-i(\d+)", url)

    if match:
        return match.group(1)

    match = re.search(r"[?&]itemId=(\d+)", url)

    if match:
        return match.group(1)

    return "unknown"


async def safe_inner_text(locator, timeout=3000, default=""):
    """
    ดึง inner_text แบบปลอดภัย
    """

    try:
        return await locator.inner_text(timeout=timeout)

    except Exception:
        return default


async def safe_get_attr(locator, attr_name, timeout=3000, default=""):
    """
    ดึง attribute แบบปลอดภัย
    """

    try:
        value = await locator.get_attribute(attr_name, timeout=timeout)
        return value if value else default

    except Exception:
        return default


async def get_product_info(page, url):
    """
    ดึงข้อมูลพื้นฐานจากหน้าสินค้า:
    - product_id
    - product_name
    - shop_name
    """

    product_id = extract_product_id_from_url(url)

    product_name_selectors = [
        "h1.pdp-mod-product-badge-title-v2",
        "h1.pdp-mod-product-badge-title",
        "h1",
    ]

    product_name = ""

    for selector in product_name_selectors:
        try:
            product_name = await page.locator(selector).first.inner_text(timeout=5000)
            product_name = product_name.strip()

            if product_name:
                break

        except Exception:
            continue

    if not product_name:
        product_name = f"Lazada Product {product_id}"

    shop_name_selectors = [
        ".seller-name-v2__detail-name",
        ".seller-name__detail-name",
        ".pdp-link.pdp-link_size_l.pdp-link_theme_black.seller-name__detail-name",
    ]

    shop_name = ""

    for selector in shop_name_selectors:
        try:
            shop_name = await page.locator(selector).first.inner_text(timeout=5000)
            shop_name = shop_name.strip()

            if shop_name:
                break

        except Exception:
            continue

    if not shop_name:
        shop_name = "Lazada Store"

    return product_id, product_name, shop_name

## Shop Key Helper

In [36]:
def extract_shop_key_from_url(shop_input):
    """
    ดึง shop_key จาก URL หรือรับ shop_key ตรง ๆ

    รองรับ:
    - mizumi-bomi
    - https://www.lazada.co.th/shop/mizumi-bomi/
    - https://www.lazada.co.th/mizumi-bomi/
    """

    if not shop_input:
        return ""

    text = str(shop_input).strip()

    if text.startswith("http"):
        match = re.search(r"lazada\.co\.th/shop/([^/?#]+)/?", text)
        if match:
            return match.group(1).strip()

        match = re.search(r"lazada\.co\.th/([^/?#]+)/?", text)
        if match:
            key = match.group(1).strip()

            if key not in ["products", "catalog"]:
                return key

        return ""

    text = text.strip("/")

    if text.startswith("shop/"):
        return text.split("/", 1)[1].strip("/")

    return text


## Product Description Helpers

In [37]:
async def wait_and_scroll_until_detail_loaded(page, max_rounds=8):
    """
    Lazada มักโหลด specs / qualification / description หลัง scroll
    """

    for round_no in range(1, max_rounds + 1):
        spec_count = await page.locator(".specification-keys .key-li").count()
        qual_count = await page.locator(".pdp-mod-qualification-items .col").count()
        desc_count = await page.locator(".detail-content").count()
        img_count = await page.locator(".detail-content img").count()

        if spec_count > 0 or qual_count > 0 or desc_count > 0 or img_count > 0:
            return {
                "spec_count": spec_count,
                "qual_count": qual_count,
                "desc_count": desc_count,
                "img_count": img_count,
                "loaded": True
            }

        await page.evaluate("window.scrollTo(0, document.body.scrollHeight * 0.35)")
        await asyncio.sleep(1.2)

        await page.evaluate("window.scrollTo(0, document.body.scrollHeight * 0.60)")
        await asyncio.sleep(1.2)

        await page.evaluate("window.scrollTo(0, document.body.scrollHeight * 0.85)")
        await asyncio.sleep(1.2)

        await page.mouse.wheel(0, 1800)
        await asyncio.sleep(1.5)

    return {
        "spec_count": await page.locator(".specification-keys .key-li").count(),
        "qual_count": await page.locator(".pdp-mod-qualification-items .col").count(),
        "desc_count": await page.locator(".detail-content").count(),
        "img_count": await page.locator(".detail-content img").count(),
        "loaded": False
    }


async def extract_product_specs(page):
    """
    ดึง specification table
    """

    specs_data = {}

    spec_rows = page.locator(".specification-keys .key-li")
    spec_count = await spec_rows.count()

    for i in range(spec_count):
        try:
            row = spec_rows.nth(i)

            key = await safe_inner_text(row.locator(".key-title"), timeout=3000)
            value = await safe_inner_text(row.locator(".key-value"), timeout=3000)

            key = clean_minimal(key)
            value = clean_minimal(value)

            if key:
                specs_data[key] = value

        except Exception:
            continue

    return specs_data


async def extract_product_qualification(page):
    """
    ดึง qualification / product highlights
    """

    qual_data = {}

    qual_cols = page.locator(".pdp-mod-qualification-items .col")
    qual_count = await qual_cols.count()

    for i in range(qual_count):
        try:
            col = qual_cols.nth(i)

            q_title = await safe_inner_text(col.locator(".title"), timeout=3000)
            q_content = await safe_inner_text(col.locator(".content"), timeout=3000)

            q_title = clean_minimal(q_title)
            q_content = clean_minimal(q_content)

            if q_title:
                qual_data[q_title] = q_content

        except Exception:
            continue

    return qual_data


async def extract_product_description(page):
    """
    ดึง description text
    """

    description_text = ""

    desc_locator = page.locator(".detail-content")

    if await desc_locator.count() > 0:
        description_text = clean_full_text(
            await safe_inner_text(desc_locator, timeout=8000)
        )

    return description_text


async def extract_product_description_images(page):
    """
    ดึงรูปภาพใน description
    """

    desc_images = []

    img_locators = page.locator(".detail-content img")
    img_count = await img_locators.count()

    for i in range(img_count):
        try:
            img = img_locators.nth(i)

            img_src = (
                await safe_get_attr(img, "src")
                or await safe_get_attr(img, "data-src")
                or await safe_get_attr(img, "data-ks-lazyload")
                or await safe_get_attr(img, "data-lazy")
                or ""
            )

            img_src = str(img_src).strip()

            if img_src and not img_src.startswith("data:"):
                desc_images.append(clean_url(img_src))

        except Exception:
            continue

    desc_images = list(dict.fromkeys(desc_images))

    return desc_images


## Listing Helper

In [38]:
def extract_is_mall(product):
    """
    ตรวจสอบว่าสินค้าเป็น LazMall หรือไม่
    """

    icons = product.get("icons", [])

    if not isinstance(icons, list):
        return False

    return any(
        isinstance(icon, dict)
        and icon.get("bizType") == "lazMall"
        for icon in icons
    )


## Row Builders

In [ ]:
# --- Product bu Shop ---

def build_product_by_shop_row(product, shop_key):
    """
    สร้าง row สำหรับ Product by Shop
    ยึด column format เดิมของ Product by Shop
    """

    price = safe_float(product.get("price"))
    original_price = safe_float(product.get("originalPrice"))
    sold_raw = product.get("itemSoldCntShow", "0")

    categories = product.get("categories", [])

    if isinstance(categories, (list, dict)):
        categories_text = json.dumps(categories, ensure_ascii=False)
    else:
        categories_text = str(categories)

    return {
        "product_id": product.get("itemId", ""),
        "sku_id": product.get("skuId", ""),
        "product_name": product.get("name", ""),
        "brand_name": clean_brand(product.get("brandName", "")),
        "categories": categories_text,
        "shop_name": product.get("sellerName", shop_key),
        "seller_id": product.get("sellerId", ""),
        "shop_key": shop_key,
        "location": product.get("location", ""),
        "current_price": price,
        "original_price": original_price,
        "sold_count": parse_sold_count(sold_raw),
        "sold_count_raw": sold_raw,
        "rating_score": safe_float(product.get("ratingScore")),
        "review_count": safe_int(product.get("review")),
        "in_stock": product.get("inStock", True),
        "is_sponsored": product.get("isSponsored", False),
        "product_url": clean_url(product.get("itemUrl", "")),
        "image_url": clean_url(product.get("image", "")),
        "source_platform": "Lazada",
        "collected_at": make_collected_at()
    }

# --- Search Keyword ---

def build_search_keyword_row(product, keyword):
    """
    สร้าง row สำหรับ Search Keyword
    ยึด column format เดิมของ Search Keyword

    หมายเหตุ:
    - ไม่มี shop_key เพราะ search result ไม่ได้มาจาก shop URL
    """

    price = safe_float(product.get("price"))
    original_price = safe_float(product.get("originalPrice"))
    sold_raw = product.get("itemSoldCntShow", "0")

    return {
        "product_id": product.get("itemId", ""),
        "sku_id": product.get("skuId", ""),
        "product_name": product.get("name", ""),
        "brand_name": clean_brand(product.get("brandName", "")),
        "discount_price": price,
        "original_price": original_price,
        "sold_count": parse_sold_count(sold_raw),
        "sold_count_raw": sold_raw,
        "rating_score": safe_float(product.get("ratingScore")),
        "review_count": safe_int(product.get("review")),
        "shop_name": product.get("sellerName", ""),
        "seller_id": product.get("sellerId", ""),
        "is_mall": extract_is_mall(product),
        "location": product.get("location", ""),
        "is_sponsored": product.get("isSponsored", False),
        "product_url": clean_url(product.get("itemUrl", "")),
        "image_url": clean_url(product.get("image", "")),
        "search_keyword": keyword,
        "source_platform": "Lazada",
        "collected_at": make_collected_at()
    }

# --- Review ---

def build_review_row(r, product_id, product_name, shop_name, product_url):
    """
    สร้าง row สำหรับ Review / Comment
    ยึด column format เดิมของ comment
    """

    review_time_raw = r.get("reviewTime", "")

    return {
        "shop_id": r.get("sellerId", ""),
        "product_id": product_id,
        "shop_name": shop_name,
        "product_name": product_name,
        "user_name": r.get("buyerName", ""),
        "rating_score": safe_float(r.get("rating", "")),
        "review_date_raw": review_time_raw,
        "review_date": convert_time_interval(review_time_raw),
        "product_option": r.get("skuInfo", ""),
        "comment_text": r.get("reviewContent", ""),
        "source_platform": "Lazada",
        "product_url": product_url,
        "collected_at": make_collected_at()
    }

# --- Product Detail ---

def build_product_detail_row(
    product_id,
    product_name,
    shop_name,
    product_url,
    qual_data,
    specs_data,
    description_text,
    desc_images
):
    """
    สร้าง row สำหรับ Product Detail / Product Description
    ยึด column format เดิมของ Product Description
    """

    return {
        "product_id": product_id,
        "product_name": product_name,
        "shop_name": shop_name,
        "product_url": product_url,
        "qualification_info": json.dumps(qual_data, ensure_ascii=False),
        "all_specs": json.dumps(specs_data, ensure_ascii=False),
        "description": description_text,
        "description_images": json.dumps(desc_images, ensure_ascii=False),
        "source_platform": "Lazada",
        "collected_at": make_collected_at()
    }

# Data Scraper

## Product by Shop

In [40]:
# ใส่ได้ทั้ง shop_key: "mizumi-bomi" หรือ URL: "https://www.lazada.co.th/shop/mizumi-bomi/"
SHOP_URL_KEYS = [
    "mizumi-bomi",
    "ing-on-official",
]

In [41]:
async def run_product_by_shop(shop_inputs):
    """
    ดึงรายการสินค้าจากหน้าร้าน Lazada ผ่าน ajax=true

    Output columns:
    - ยึดตาม Product by Shop format เดิม
    - shop_key มาจาก URL ร้าน หรือ shop_key ที่ใส่
    """

    scraper_name = "product_by_shop"

    if RESUME_FROM_BACKUP:
        all_products = load_existing_backup(
            data_prefix="temp_lazada_products",
            dedupe_cols=PRODUCT_DEDUPE_COLS
        )
    else:
        all_products = []

    product_log_rows = []
    failed_shops = []

    seen_products = {
        (
            str(row.get("product_id", "")),
            str(row.get("sku_id", "")),
            str(row.get("product_url", ""))
        )
        for row in all_products
    }

    async with async_playwright() as p:
        print(f"📂 [SESSION] Loading Persistent Context: '{USER_DATA_DIR}'")

        context, page = await create_browser_context(
            p,
            block_images=True
        )

        try:
            for idx, shop_input in enumerate(shop_inputs, start=1):
                shop_key = extract_shop_key_from_url(shop_input)

                print("\n" + "-" * 60)
                print(f"[{idx}/{len(shop_inputs)}] Processing Shop: {shop_key}")

                if not shop_key:
                    failed_shops.append({
                        "shop_input": shop_input,
                        "error": "Cannot extract shop_key"
                    })

                    add_scrape_log(
                        product_log_rows,
                        scraper_name=scraper_name,
                        level="ERROR",
                        message="Cannot extract shop_key",
                        shop_input=shop_input
                    )

                    continue

                shop_products = []

                base_url = (
                    f"https://www.lazada.co.th/{shop_key}/"
                    "?ajax=true"
                    "&from=wangpu"
                    "&isFirstRequest=true"
                    "&langFlag=th"
                    "&page=1"
                    "&pageTypeId=2"
                    "&q=All-Products"
                )

                try:
                    await page.goto(
                        base_url,
                        wait_until="domcontentloaded",
                        timeout=60000
                    )

                    raw_text = await handle_page_check(
                        page,
                        f"Shop Meta: {shop_key}"
                    )

                    data = safe_json_loads(raw_text)

                    if not data:
                        failed_shops.append({
                            "shop_key": shop_key,
                            "error": "Empty JSON or JSON Decode Error at shop meta"
                        })

                        add_scrape_log(
                            product_log_rows,
                            scraper_name=scraper_name,
                            level="ERROR",
                            message="Empty JSON or JSON Decode Error at shop meta",
                            shop_key=shop_key
                        )

                        save_scrape_checkpoint(
                            rows=all_products,
                            data_prefix="temp_lazada_products",
                            dedupe_cols=PRODUCT_DEDUPE_COLS,
                            reason="shop meta error"
                        )

                        continue

                    main_info = data.get("mainInfo", {}) or {}

                    total_results = safe_int(main_info.get("totalResults", 0))
                    page_size = safe_int(main_info.get("pageSize", 40), default=40)

                    total_pages = (
                        math.ceil(total_results / page_size)
                        if total_results > 0 and page_size > 0
                        else 1
                    )

                    print(
                        f"   > Total Products Found: {total_results:,} | "
                        f"Total Pages: {total_pages:,}"
                    )

                    for page_no in range(1, total_pages + 1):
                        current_url = base_url.replace(
                            "page=1",
                            f"page={page_no}"
                        )

                        await page.goto(
                            current_url,
                            wait_until="domcontentloaded",
                            timeout=60000
                        )

                        await human_delay()

                        page_raw = await handle_page_check(
                            page,
                            f"Shop: {shop_key} | Page {page_no}"
                        )

                        page_json = safe_json_loads(page_raw)

                        if not page_json:
                            failed_shops.append({
                                "shop_key": shop_key,
                                "page_no": page_no,
                                "error": "JSON Decode Error at shop page"
                            })

                            add_scrape_log(
                                product_log_rows,
                                scraper_name=scraper_name,
                                level="ERROR",
                                message="JSON Decode Error at shop page",
                                shop_key=shop_key,
                                page_no=page_no,
                                collected_rows=len(all_products)
                            )

                            save_scrape_checkpoint(
                                rows=all_products,
                                data_prefix="temp_lazada_products",
                                dedupe_cols=PRODUCT_DEDUPE_COLS,
                                reason="shop page json error"
                            )

                            break

                        items = (
                            page_json
                            .get("mods", {})
                            .get("listItems", [])
                        ) or []

                        print(
                            f"   > Page {page_no}/{total_pages}: "
                            f"Found {len(items)} products"
                        )

                        if not items:
                            add_scrape_log(
                                product_log_rows,
                                scraper_name=scraper_name,
                                level="WARNING",
                                message="No items found at shop page",
                                shop_key=shop_key,
                                page_no=page_no
                            )

                            break

                        current_page_products = []

                        for product in items:
                            row = build_product_by_shop_row(
                                product=product,
                                shop_key=shop_key
                            )

                            row_key = (
                                str(row.get("product_id", "")),
                                str(row.get("sku_id", "")),
                                str(row.get("product_url", ""))
                            )

                            if row_key in seen_products:
                                continue

                            current_page_products.append(row)
                            seen_products.add(row_key)

                        shop_products.extend(current_page_products)
                        all_products.extend(current_page_products)

                        if page_no % PRODUCT_CHECKPOINT_EVERY_PAGES == 0:
                            add_scrape_log(
                                product_log_rows,
                                scraper_name=scraper_name,
                                level="INFO",
                                message="Product checkpoint saved",
                                shop_key=shop_key,
                                page_no=page_no,
                                collected_rows=len(all_products)
                            )

                            save_scrape_checkpoint(
                                rows=all_products,
                                data_prefix="temp_lazada_products",
                                dedupe_cols=PRODUCT_DEDUPE_COLS,
                                reason=f"every {PRODUCT_CHECKPOINT_EVERY_PAGES} pages"
                            )

                    print(
                        f"   >>> Successfully collected "
                        f"{len(shop_products):,} new products from '{shop_key}'"
                    )

                except Exception as e:
                    print(f"❌ [SKIP] Error in shop '{shop_key}': {e}")

                    failed_shops.append({
                        "shop_key": shop_key,
                        "error": str(e)
                    })

                    add_scrape_log(
                        product_log_rows,
                        scraper_name=scraper_name,
                        level="ERROR",
                        message="Error while scraping shop",
                        shop_key=shop_key,
                        error=str(e),
                        collected_rows=len(all_products)
                    )

                    save_scrape_checkpoint(
                        rows=all_products,
                        data_prefix="temp_lazada_products",
                        dedupe_cols=PRODUCT_DEDUPE_COLS,
                        reason="shop error"
                    )

                    continue

        finally:
            save_scrape_checkpoint(
                rows=all_products,
                data_prefix="temp_lazada_products",
                dedupe_cols=PRODUCT_DEDUPE_COLS,
                reason="final backup before browser close"
            )

            await context.close()

    df_products, filename = save_final_data(
        rows=all_products,
        final_prefix="lazada_product",
        dedupe_cols=PRODUCT_DEDUPE_COLS
    )

    save_failed_log(
        failed_rows=failed_shops,
        prefix="failed_shops",
        scraper_name=scraper_name
    )

    return df_products

df_products = await run_product_by_shop(SHOP_URL_KEYS)

ℹ️ No existing temp backup found: collected_data\temp\temp_lazada_products.csv or collected_data\temp\temp_lazada_products.xlsx
📂 [SESSION] Loading Persistent Context: './my_session'

------------------------------------------------------------
[1/2] Processing Shop: mizumi-bomi
   > Total Products Found: 173 | Total Pages: 5
   > Page 1/5: Found 40 products
   > Page 2/5: Found 40 products
   > Page 3/5: Found 40 products
   > Page 4/5: Found 40 products
   > Page 5/5: Found 13 products
   >>> Successfully collected 173 new products from 'mizumi-bomi'

------------------------------------------------------------
[2/2] Processing Shop: ing-on-official
   > Total Products Found: 56 | Total Pages: 2
   > Page 1/2: Found 40 products
   > Page 2/2: Found 16 products
   >>> Successfully collected 56 new products from 'ing-on-official'
   💾 Temp checkpoint saved: collected_data\temp\temp_lazada_products.csv | rows=229 | reason=final backup before browser close

✅ FINAL FILE SAVED: lazada_pro

In [42]:
pp.pprint(df_products.to_dict(orient="records"))

[ { 'product_id': '265012126',
    'sku_id': '653866412',
    'product_name': '[มีแพ็ค 2 และ 4 หลอดให้เลือก] MizuMi UV Water Serum '
                    'SPF50+ PA++++ 40g  No.1 Best Selling Sunscreen '
                    'เซรั่มกันแดด บางเบา ซึมไว ไม่อุดตัน',
    'brand_name': 'MizuMi',
    'categories': '[3838, 4079, 5885]',
    'shop_name': 'MizuMi & Bomi',
    'seller_id': '100152243',
    'shop_key': 'mizumi-bomi',
    'location': 'Samut Prakan',
    'current_price': 890.0,
    'original_price': 1780.0,
    'sold_count': 100200,
    'sold_count_raw': '100.2K sold',
    'rating_score': 4.983807744440781,
    'review_count': 24320,
    'in_stock': True,
    'is_sponsored': False,
    'product_url': 'https://www.lazada.co.th/products/pdp-i265012126.html',
    'image_url': 'https://th-live-01.slatic.net/p/40fb5374c2493bbc87dd08a2da952e60.jpg',
    'source_platform': 'Lazada',
    'collected_at': '2026-05-04 13:08:21'},
  { 'product_id': '16152011063',
    'sku_id': '127137723225',
  

## Product Description

In [43]:
DETAIL_PRODUCT_URLS = [
    "https://www.lazada.co.th/products/pdp-i5010918982-s21176047921.html",
    "https://www.lazada.co.th/products/pdp-i4599133168-s18943395331.html",
]

In [44]:
async def run_product_detail(product_urls):
    """
    ดึงรายละเอียดสินค้า Lazada

    Output columns:
    - ยึดตาม Product Description format เดิม
    """

    scraper_name = "product_detail"

    if RESUME_FROM_BACKUP:
        all_details = load_existing_backup(
            data_prefix="temp_lazada_details",
            dedupe_cols=DETAIL_DEDUPE_COLS
        )
    else:
        all_details = []

    detail_log_rows = []
    failed_details = []

    done_urls = {
        str(row.get("product_url", ""))
        for row in all_details
        if str(row.get("product_url", ""))
    }

    async with async_playwright() as p:
        print(f"📂 [SESSION] Loading Persistent Context: '{USER_DATA_DIR}'")

        context, page = await create_browser_context(
            p,
            block_images=False
        )

        try:
            for idx, url in enumerate(product_urls, start=1):
                if url in done_urls:
                    print(f"⏭️ Skip already scraped detail URL: {url}")
                    continue

                print("\n" + "-" * 60)
                print(f"[{idx}/{len(product_urls)}] Product Detail URL: {url}")

                try:
                    await page.goto(
                        url,
                        wait_until="domcontentloaded",
                        timeout=90000
                    )

                    await handle_page_check(
                        page,
                        f"Product Detail: {idx}"
                    )

                    try:
                        await page.wait_for_load_state(
                            "networkidle",
                            timeout=90000
                        )

                    except Exception as e:
                        add_scrape_log(
                            detail_log_rows,
                            scraper_name=scraper_name,
                            level="WARNING",
                            message="Networkidle timeout, continue scraping detail",
                            product_url=url,
                            url_index=idx,
                            warning=str(e)
                        )

                    product_id, product_name, shop_name = await get_product_info(
                        page,
                        url
                    )

                    load_result = await wait_and_scroll_until_detail_loaded(page)

                    if not load_result["loaded"]:
                        add_scrape_log(
                            detail_log_rows,
                            scraper_name=scraper_name,
                            level="WARNING",
                            message="Detail content may not be fully loaded",
                            product_url=url,
                            product_id=product_id,
                            url_index=idx
                        )

                    specs_data = await extract_product_specs(page)
                    qual_data = await extract_product_qualification(page)
                    description_text = await extract_product_description(page)
                    desc_images = await extract_product_description_images(page)

                    print(f"   > Product ID: {product_id}")
                    print(f"   > Product Name: {product_name}")
                    print(f"   > Specs: {len(specs_data)} keys")
                    print(f"   > Qualifications: {len(qual_data)} keys")
                    print(f"   > Images: {len(desc_images)}")
                    print(f"   > Description: {len(description_text)} chars")

                    if (
                        len(specs_data) == 0
                        and len(qual_data) == 0
                        and len(description_text) == 0
                        and len(desc_images) == 0
                    ):
                        html = await page.content()
                        save_debug_html(
                            html,
                            "debug_empty_detail",
                            product_id
                        )

                        add_scrape_log(
                            detail_log_rows,
                            scraper_name=scraper_name,
                            level="WARNING",
                            message="Empty product detail content",
                            product_url=url,
                            product_id=product_id,
                            url_index=idx
                        )

                    row = build_product_detail_row(
                        product_id=product_id,
                        product_name=product_name,
                        shop_name=shop_name,
                        product_url=url,
                        qual_data=qual_data,
                        specs_data=specs_data,
                        description_text=description_text,
                        desc_images=desc_images
                    )

                    all_details.append(row)
                    done_urls.add(url)

                    if idx % DETAIL_CHECKPOINT_EVERY_URLS == 0:
                        add_scrape_log(
                            detail_log_rows,
                            scraper_name=scraper_name,
                            level="INFO",
                            message="Product detail checkpoint saved",
                            product_url=url,
                            url_index=idx,
                            collected_rows=len(all_details)
                        )

                        save_scrape_checkpoint(
                            rows=all_details,
                            data_prefix="temp_lazada_details",
                            dedupe_cols=DETAIL_DEDUPE_COLS,
                            reason=f"every {DETAIL_CHECKPOINT_EVERY_URLS} urls"
                        )

                    await human_delay()

                except Exception as e:
                    print(f"   ❌ [ERROR] Product detail failed: {e}")

                    failed_details.append({
                        "product_url": url,
                        "url_index": idx,
                        "error": str(e)
                    })

                    add_scrape_log(
                        detail_log_rows,
                        scraper_name=scraper_name,
                        level="ERROR",
                        message="Error while scraping product detail",
                        product_url=url,
                        url_index=idx,
                        error=str(e),
                        collected_rows=len(all_details)
                    )

                    save_scrape_checkpoint(
                        rows=all_details,
                        data_prefix="temp_lazada_details",
                        dedupe_cols=DETAIL_DEDUPE_COLS,
                        reason="product detail error"
                    )

                    continue

        finally:
            save_scrape_checkpoint(
                rows=all_details,
                data_prefix="temp_lazada_details",
                dedupe_cols=DETAIL_DEDUPE_COLS,
                reason="final backup before browser close"
            )

            await context.close()

    df_details, filename = save_final_data(
        rows=all_details,
        final_prefix="lazada_detail",
        dedupe_cols=DETAIL_DEDUPE_COLS
    )

    save_failed_log(
        failed_rows=failed_details,
        prefix="failed_details",
        scraper_name=scraper_name
    )

    return df_details

df_details = await run_product_detail(DETAIL_PRODUCT_URLS)

ℹ️ No existing temp backup found: collected_data\temp\temp_lazada_details.csv or collected_data\temp\temp_lazada_details.xlsx
📂 [SESSION] Loading Persistent Context: './my_session'

------------------------------------------------------------
[1/2] Product Detail URL: https://www.lazada.co.th/products/pdp-i5010918982-s21176047921.html
   > Product ID: 5010918982
   > Product Name: [แพ็ค 3] Bomi Bio S Series เซตทดลอง ดูแลน้ำหนัก สุขภาพดี พร้อมเพิ่มกากใย ลำไส้สมดุล
   > Specs: 7 keys
   > Qualifications: 2 keys
   > Images: 5
   > Description: 2379 chars

------------------------------------------------------------
[2/2] Product Detail URL: https://www.lazada.co.th/products/pdp-i4599133168-s18943395331.html
   > Product ID: 4599133168
   > Product Name: SRICHAND FEELIN’ ME MATTE LIQUID LIP (3 g)
   > Specs: 9 keys
   > Qualifications: 2 keys
   > Images: 0
   > Description: 990 chars
   💾 Temp checkpoint saved: collected_data\temp\temp_lazada_details.csv | rows=2 | reason=final backup be

In [45]:
pp.pprint(df_details.to_dict(orient="records"))

[ { 'product_id': '5010918982',
    'product_name': '[แพ็ค 3] Bomi Bio S Series เซตทดลอง ดูแลน้ำหนัก สุขภาพดี '
                    'พร้อมเพิ่มกากใย ลำไส้สมดุล',
    'shop_name': 'MizuMi & Bomi',
    'product_url': 'https://www.lazada.co.th/products/pdp-i5010918982-s21176047921.html',
    'qualification_info': '{"License Type": "TH_FDA_Advertising", "License '
                          'Code": ""}',
    'all_specs': '{"Brand": "Bomi", "SKU": "5010918982_TH-21176047921", '
                 '"Product_License": "20-1-13451-6-0001", "Product Form": '
                 '"Liquid/Powder", "Ingredients": "Fiber", "Pack Type": '
                 '"Multi-pack", "Recommended User": "Adults"}',
    'description': 'Bomi Coffee Bio S\n'
                   'โบมิ คอฟฟี่ ไบโอ เอส\n'
                   'กาแฟคุมน้ำหนัก หอมอร่อย มีพรีไบโอติกส์ไฟเบอร์สูง\n'
                   '✔️ High Prebiotic Fiber '
                   'ไฟเบอร์พรีไบโอติกส์ชั้นดีในปริมาณสูง '
                   'ช่วยเพิ่มปริมาณโพรไบโอติกส์

## Reviews

In [46]:
REVIEW_PRODUCT_URLS = [
    "https://www.lazada.co.th/products/pdp-i5592245338.html?spm=a2o4m.store_keyword.list.1.4bfd58a6YLqDCI",
    "https://www.lazada.co.th/products/pdp-i5592124928.html?spm=a2o4m.store_keyword.list.3.4bfd58a6YLqDCI",
]

In [49]:
async def run_reviews(product_urls):
    """
    ดึงข้อมูลรีวิวสินค้าจาก Lazada Review API

    Output columns:
    - ยึดตาม comment format เดิม
    """

    scraper_name = "reviews"

    if RESUME_FROM_BACKUP:
        all_reviews = load_existing_backup(
            data_prefix="temp_lazada_reviews",
            dedupe_cols=REVIEW_DEDUPE_COLS
        )
    else:
        all_reviews = []

    review_log_rows = []
    failed_products = []

    seen_reviews = {
        (
            str(row.get("product_id", "")),
            str(row.get("user_name", "")),
            str(row.get("review_date_raw", "")),
            str(row.get("comment_text", ""))
        )
        for row in all_reviews
    }

    async with async_playwright() as p:
        print(f"📂 [SESSION] Loading Persistent Context: '{USER_DATA_DIR}'")

        context, page = await create_browser_context(
            p,
            block_images=True
        )

        try:
            for idx, url in enumerate(product_urls, start=1):
                print("\n" + "-" * 60)
                print(f"[{idx}/{len(product_urls)}] Processing Review URL: {url}")

                product_reviews = []

                try:
                    await page.goto(
                        url,
                        wait_until="domcontentloaded",
                        timeout=60000
                    )

                    await handle_page_check(
                        page,
                        f"Product Page: {idx}"
                    )

                    product_id, product_name, shop_name = await get_product_info(
                        page,
                        url
                    )

                    print(f"   > Product ID: {product_id}")
                    print(f"   > Product Name: {product_name}")
                    print(f"   > Shop Name: {shop_name}")

                    if product_id == "unknown":
                        raise ValueError("Cannot extract product_id from URL")

                    page_no = 1
                    total_pages = None

                    while True:
                        review_api_url = (
                            "https://my.lazada.co.th/pdp/review/getReviewList"
                            f"?itemId={product_id}"
                            f"&pageSize=50"
                            f"&filter=0"
                            f"&sort=0"
                            f"&pageNo={page_no}"
                        )

                        await page.goto(
                            review_api_url,
                            wait_until="domcontentloaded",
                            timeout=60000
                        )

                        await human_delay()

                        raw_text = await handle_page_check(
                            page,
                            f"Review API | Product {idx} | Page {page_no}"
                        )

                        data = safe_json_loads(raw_text)

                        if not data:
                            failed_products.append({
                                "product_url": url,
                                "product_id": product_id,
                                "page_no": page_no,
                                "error": "JSON Decode Error at review page"
                            })

                            add_scrape_log(
                                review_log_rows,
                                scraper_name=scraper_name,
                                level="ERROR",
                                message="JSON Decode Error at review page",
                                product_url=url,
                                product_id=product_id,
                                page_no=page_no,
                                collected_rows=len(all_reviews)
                            )

                            save_scrape_checkpoint(
                                rows=all_reviews,
                                data_prefix="temp_lazada_reviews",
                                dedupe_cols=REVIEW_DEDUPE_COLS,
                                reason="review json error"
                            )

                            break

                        model = data.get("model", {}) or {}
                        items = model.get("items", []) or []

                        if not items:
                            print(f"   > Page {page_no}: No more reviews.")
                            break

                        if total_pages is None:
                            review_count = safe_int(
                                model.get("ratings", {}).get("reviewCount", 0)
                            )

                            if review_count > 0:
                                total_pages = math.ceil(review_count / 50)

                        current_page_reviews = []

                        for r in items:
                            row = build_review_row(
                                r=r,
                                product_id=product_id,
                                product_name=product_name,
                                shop_name=shop_name,
                                product_url=url
                            )

                            review_key = (
                                str(row.get("product_id", "")),
                                str(row.get("user_name", "")),
                                str(row.get("review_date_raw", "")),
                                str(row.get("comment_text", ""))
                            )

                            if review_key in seen_reviews:
                                continue

                            current_page_reviews.append(row)
                            seen_reviews.add(review_key)

                        product_reviews.extend(current_page_reviews)
                        all_reviews.extend(current_page_reviews)

                        print(f"   > Page {page_no}: Found {len(current_page_reviews)} reviews | Total collected: {len(product_reviews):,}")

                        if page_no % REVIEW_CHECKPOINT_EVERY_PAGES == 0:
                            add_scrape_log(
                                review_log_rows,
                                scraper_name=scraper_name,
                                level="INFO",
                                message="Review checkpoint saved",
                                product_id=product_id,
                                product_url=url,
                                page_no=page_no,
                                collected_rows=len(all_reviews)
                            )

                            save_scrape_checkpoint(
                                rows=all_reviews,
                                data_prefix="temp_lazada_reviews",
                                dedupe_cols=REVIEW_DEDUPE_COLS,
                                reason=f"every {REVIEW_CHECKPOINT_EVERY_PAGES} pages"
                            )

                        if total_pages is not None and page_no >= total_pages:
                            print("   > Reached total review pages. Stop.")
                            break

                        page_no += 1

                    print(f"   >>> Successfully collected {len(product_reviews):,} new reviews for this product.")

                except Exception as e:
                    print(f"   ❌ Error processing review URL: {e}")

                    failed_products.append({
                        "product_url": url,
                        "error": str(e)
                    })

                    add_scrape_log(
                        review_log_rows,
                        scraper_name=scraper_name,
                        level="ERROR",
                        message="Error while scraping reviews",
                        product_url=url,
                        error=str(e),
                        collected_rows=len(all_reviews)
                    )

                    save_scrape_checkpoint(
                        rows=all_reviews,
                        data_prefix="temp_lazada_reviews",
                        dedupe_cols=REVIEW_DEDUPE_COLS,
                        reason="review error"
                    )

                    continue

        finally:
            save_scrape_checkpoint(
                rows=all_reviews,
                data_prefix="temp_lazada_reviews",
                dedupe_cols=REVIEW_DEDUPE_COLS,
                reason="final backup before browser close"
            )

            await context.close()

    df_reviews, filename = save_final_data(
        rows=all_reviews,
        final_prefix="lazada_review",
        dedupe_cols=REVIEW_DEDUPE_COLS
    )

    save_failed_log(
        failed_rows=failed_products,
        prefix="failed_reviews",
        scraper_name=scraper_name
    )

    return df_reviews

df_reviews = await run_reviews(REVIEW_PRODUCT_URLS)

🔁 Existing temp backup loaded: collected_data\temp\temp_lazada_reviews.csv | rows=100
📂 [SESSION] Loading Persistent Context: './my_session'

------------------------------------------------------------
[1/2] Processing Review URL: https://www.lazada.co.th/products/pdp-i5592245338.html?spm=a2o4m.store_keyword.list.1.4bfd58a6YLqDCI
   > Product ID: 5592245338
   > Product Name: MizuMi UV Bright Body Serum Beige (120g) เซรั่มกันแดดโทนอัพ ผิวไบรท์ทันที ปรับผิวกระจ่างใสขึ้น 1 ระดับสำหรับผิวขาวเหลือง-ผิวสองสี
   > Shop Name: MizuMi & Bomi
   > Page 1: Found 0 reviews | Total collected: 0
   > Page 2: Found 3 reviews | Total collected: 3
   > Page 3: No more reviews.
   >>> Successfully collected 3 new reviews for this product.

------------------------------------------------------------
[2/2] Processing Review URL: https://www.lazada.co.th/products/pdp-i5592124928.html?spm=a2o4m.store_keyword.list.3.4bfd58a6YLqDCI
   > Product ID: 5592124928
   > Product Name: [แพ็คคู่] MizuMi UV Bright Bo

In [50]:
pp.pprint(df_reviews.to_dict(orient="records"))

[ { 'shop_id': 100152243,
    'product_id': 5592245338,
    'shop_name': 'MizuMi & Bomi',
    'product_name': 'MizuMi UV Bright Body Serum Beige (120g) '
                    'เซรั่มกันแดดโทนอัพ ผิวไบรท์ทันที ปรับผิวกระจ่างใสขึ้น 1 '
                    'ระดับสำหรับผิวขาวเหลือง-ผิวสองสี',
    'user_name': 'J***i',
    'rating_score': 5.0,
    'review_date_raw': '05 Apr 2025',
    'review_date': '2025-04-05',
    'product_option': 'Variation3:Beige',
    'comment_text': 'ผิวสีเหลืองใช้อันนี้แล้วผิวสว่างขึ้นแบบธรรมชาติ '
                    'ระวังอย่าทาเยอะเกินค่า\n'
                    ' 💧Moisturizing Effect: ไม่หนักผิวเลย เบาสบาย ',
    'source_platform': 'Lazada',
    'product_url': 'https://www.lazada.co.th/products/pdp-i5592245338.html?spm=a2o4m.store_keyword.list.1.4bfd58a6YLqDCI',
    'collected_at': '2026-05-04 13:10:51'},
  { 'shop_id': 100152243,
    'product_id': 5592245338,
    'shop_name': 'MizuMi & Bomi',
    'product_name': 'MizuMi UV Bright Body Serum Beige (120g) '
      

## Search Keyword

In [51]:
SEARCH_KEYWORDS = [
    "ครีมกันแดด",
    "มาสก์หน้า",
    # "รองพื้น",
    # "อายไลเนอร์",
    # "บลัชออน",
    # "แป้งพัฟ",
    # "คลีนซิ่ง",
    # "คลีนเซอร์",
    # "eye shadow",
    # "toner pad"
]

SEARCH_OFFICIAL_ONLY = False

In [52]:
async def run_search_keyword(
    keywords,
    official_only=SEARCH_OFFICIAL_ONLY,
    start_page_by_keyword=None,
    end_page_by_keyword=None
):
    """
    ดึงรายการสินค้าจากหน้า search Lazada ผ่าน ajax=true

    Output columns:
    - ยึดตาม Search Keyword format เดิม
    - ไม่มี shop_key เพราะ search result ไม่ได้มาจาก shop URL

    Optional resume:
    start_page_by_keyword={"อายไลเนอร์": 59}
    end_page_by_keyword={"อายไลเนอร์": 102}
    """

    scraper_name = "search_keyword"

    if start_page_by_keyword is None:
        start_page_by_keyword = {}

    if end_page_by_keyword is None:
        end_page_by_keyword = {}

    if RESUME_FROM_BACKUP:
        all_products = load_existing_backup(
            data_prefix="temp_lazada_search",
            dedupe_cols=SEARCH_DEDUPE_COLS
        )
    else:
        all_products = []

    search_log_rows = []
    failed_keywords = []

    seen_search_products = {
        (
            str(row.get("search_keyword", "")),
            str(row.get("product_id", "")),
            str(row.get("sku_id", "")),
            str(row.get("product_url", ""))
        )
        for row in all_products
    }

    async with async_playwright() as p:
        print(f"📂 [SESSION] Loading Persistent Context: '{USER_DATA_DIR}'")

        context, page = await create_browser_context(
            p,
            block_images=True
        )

        try:
            for idx, keyword in enumerate(keywords, start=1):
                keyword_products = []
                encoded_keyword = quote_plus(keyword)
                official_param = "&service=official" if official_only else ""

                print("\n" + "-" * 60)
                print(
                    f"[{idx}/{len(keywords)}] "
                    f"Search Keyword: {keyword} | LazMall: {official_only}"
                )

                base_url = (
                    "https://www.lazada.co.th/catalog/"
                    f"?ajax=true"
                    f"&q={encoded_keyword}"
                    f"&page=1"
                    f"&sort=sold"
                    f"{official_param}"
                )

                try:
                    await page.goto(
                        "https://www.lazada.co.th/",
                        wait_until="domcontentloaded",
                        timeout=60000
                    )

                    await asyncio.sleep(1.5)

                    await page.goto(
                        base_url,
                        wait_until="domcontentloaded",
                        timeout=60000
                    )

                    raw_text = await handle_page_check(
                        page,
                        f"Search Meta: {keyword}"
                    )

                    data = safe_json_loads(raw_text)

                    if not data:
                        failed_keywords.append({
                            "keyword": keyword,
                            "error": "Empty JSON or JSON Decode Error at search meta"
                        })

                        add_scrape_log(
                            search_log_rows,
                            scraper_name=scraper_name,
                            level="ERROR",
                            message="Empty JSON or JSON Decode Error at search meta",
                            keyword=keyword,
                            collected_rows=len(all_products)
                        )

                        save_scrape_checkpoint(
                            rows=all_products,
                            data_prefix="temp_lazada_search",
                            dedupe_cols=SEARCH_DEDUPE_COLS,
                            reason="search meta error"
                        )

                        continue

                    main_info = data.get("mainInfo", {}) or {}

                    total_results = safe_int(main_info.get("totalResults", 0))
                    
                    # page_size = safe_int(main_info.get("pageSize", 40), default=40)

                    # total_pages = (
                    #     math.ceil(total_results / page_size)
                    #     if total_results > 0 and page_size > 0
                    #     else 1
                    # )

                    total_pages = 5

                    start_page = int(start_page_by_keyword.get(keyword, 1))
                    end_page = int(end_page_by_keyword.get(keyword, total_pages))
                    end_page = min(end_page, total_pages)

                    print(
                        f"   > Total Items Found: {total_results:,} | "
                        f"Total Pages: {total_pages:,}"
                    )

                    print(
                        f"   > Page Range to Scrape: "
                        f"{start_page:,} → {end_page:,}"
                    )

                    if start_page > end_page:
                        add_scrape_log(
                            search_log_rows,
                            scraper_name=scraper_name,
                            level="WARNING",
                            message="Skip keyword because start_page > end_page",
                            keyword=keyword,
                            start_page=start_page,
                            end_page=end_page
                        )

                        continue

                    for page_no in range(start_page, end_page + 1):
                        current_url = base_url.replace(
                            "page=1",
                            f"page={page_no}"
                        )

                        await page.goto(
                            current_url,
                            wait_until="domcontentloaded",
                            timeout=60000
                        )

                        await human_delay()

                        page_raw_text = await handle_page_check(
                            page,
                            f"Search: {keyword} | Page {page_no}"
                        )

                        page_json = safe_json_loads(page_raw_text)

                        if not page_json:
                            failed_keywords.append({
                                "keyword": keyword,
                                "page_no": page_no,
                                "error": "JSON Decode Error at search page"
                            })

                            add_scrape_log(
                                search_log_rows,
                                scraper_name=scraper_name,
                                level="ERROR",
                                message="JSON Decode Error at search page",
                                keyword=keyword,
                                page_no=page_no,
                                collected_rows=len(all_products)
                            )

                            save_scrape_checkpoint(
                                rows=all_products,
                                data_prefix="temp_lazada_search",
                                dedupe_cols=SEARCH_DEDUPE_COLS,
                                reason="search page json error"
                            )

                            break

                        items = (
                            page_json
                            .get("mods", {})
                            .get("listItems", [])
                        ) or []

                        print(
                            f"   > Page {page_no}/{end_page}: "
                            f"Found {len(items)} items"
                        )

                        if not items:
                            add_scrape_log(
                                search_log_rows,
                                scraper_name=scraper_name,
                                level="WARNING",
                                message="No items found at search page",
                                keyword=keyword,
                                page_no=page_no,
                                collected_rows=len(all_products)
                            )

                            break

                        current_page_products = []

                        for product in items:
                            row = build_search_keyword_row(
                                product=product,
                                keyword=keyword
                            )

                            row_key = (
                                str(row.get("search_keyword", "")),
                                str(row.get("product_id", "")),
                                str(row.get("sku_id", "")),
                                str(row.get("product_url", ""))
                            )

                            if row_key in seen_search_products:
                                continue

                            current_page_products.append(row)
                            seen_search_products.add(row_key)

                        keyword_products.extend(current_page_products)
                        all_products.extend(current_page_products)

                        pages_done = page_no - start_page + 1

                        if (
                            pages_done % SEARCH_CHECKPOINT_EVERY_PAGES == 0
                            or page_no == end_page
                        ):
                            add_scrape_log(
                                search_log_rows,
                                scraper_name=scraper_name,
                                level="INFO",
                                message="Search keyword checkpoint saved",
                                keyword=keyword,
                                page_no=page_no,
                                start_page=start_page,
                                end_page=end_page,
                                collected_rows=len(all_products)
                            )

                            save_scrape_checkpoint(
                                rows=all_products,
                                data_prefix="temp_lazada_search",
                                dedupe_cols=SEARCH_DEDUPE_COLS,
                                reason=f"checkpoint at page {page_no}"
                            )

                    print(
                        f"   >>> Successfully collected "
                        f"{len(keyword_products):,} new items for keyword '{keyword}'."
                    )

                except Exception as e:
                    print(f"❌ [SKIP] Error in keyword '{keyword}': {e}")

                    failed_keywords.append({
                        "keyword": keyword,
                        "error": str(e)
                    })

                    add_scrape_log(
                        search_log_rows,
                        scraper_name=scraper_name,
                        level="ERROR",
                        message="Error while scraping keyword",
                        keyword=keyword,
                        error=str(e),
                        collected_rows=len(all_products)
                    )

                    save_scrape_checkpoint(
                        rows=all_products,
                        data_prefix="temp_lazada_search",
                        dedupe_cols=SEARCH_DEDUPE_COLS,
                        reason="keyword error"
                    )

                    continue

        finally:
            save_scrape_checkpoint(
                rows=all_products,
                data_prefix="temp_lazada_search",
                dedupe_cols=SEARCH_DEDUPE_COLS,
                reason="final backup before browser close"
            )

            await context.close()

    df_search, filename = save_final_data(
        rows=all_products,
        final_prefix="lazada_search",
        dedupe_cols=SEARCH_DEDUPE_COLS
    )

    save_failed_log(
        failed_rows=failed_keywords,
        prefix="failed_keywords",
        scraper_name=scraper_name
    )

    return df_search

In [53]:
# Run ปกติ
df_search = await run_search_keyword(
    SEARCH_KEYWORDS,
    official_only=SEARCH_OFFICIAL_ONLY
)


# Run ต่อเฉพาะ keyword/page ที่ error
# df_search = await run_search_keyword(
#     ["อายไลเนอร์"],
#     official_only=SEARCH_OFFICIAL_ONLY,
#     start_page_by_keyword={"อายไลเนอร์": 59},
#     end_page_by_keyword={"อายไลเนอร์": 102}
# )

ℹ️ No existing temp backup found: collected_data\temp\temp_lazada_search.csv or collected_data\temp\temp_lazada_search.xlsx
📂 [SESSION] Loading Persistent Context: './my_session'

------------------------------------------------------------
[1/2] Search Keyword: ครีมกันแดด | LazMall: False
   > Total Items Found: 4,080 | Total Pages: 5
   > Page Range to Scrape: 1 → 5
   > Page 1/5: Found 40 items
   > Page 2/5: Found 40 items
   > Page 3/5: Found 40 items
   > Page 4/5: Found 40 items
   > Page 5/5: Found 40 items
   ℹ️ [search_keyword] Search keyword checkpoint saved
   💾 Temp checkpoint saved: collected_data\temp\temp_lazada_search.csv | rows=200 | reason=checkpoint at page 5
   >>> Successfully collected 200 new items for keyword 'ครีมกันแดด'.

------------------------------------------------------------
[2/2] Search Keyword: มาสก์หน้า | LazMall: False
   > Total Items Found: 4,080 | Total Pages: 5
   > Page Range to Scrape: 1 → 5
   > Page 1/5: Found 40 items
   > Page 2/5: Found 

In [54]:
pp.pprint(df_search.to_dict(orient="records"))

[ { 'product_id': '5788089868',
    'sku_id': '24639484981',
    'product_name': 'Eucerin Sun Protection Sun serum spotless brightening '
                    'SPF50 + PA +++ 50 ml',
    'brand_name': 'Eucerin',
    'discount_price': 248.3,
    'original_price': 300.0,
    'sold_count': 440,
    'sold_count_raw': '440 sold',
    'rating_score': 4.96078431372549,
    'review_count': 153,
    'shop_name': 'Luminaia',
    'seller_id': '101122736328',
    'is_mall': False,
    'location': 'Bangkok',
    'is_sponsored': False,
    'product_url': 'https://www.lazada.co.th/products/pdp-i5788089868.html',
    'image_url': 'https://th-live-01.slatic.net/p/83cad6b13fc5022c1aedc1e5f304bb64.jpg',
    'search_keyword': 'ครีมกันแดด',
    'source_platform': 'Lazada',
    'collected_at': '2026-05-04 13:18:01'},
  { 'product_id': '5367365709',
    'sku_id': '22816057014',
    'product_name': '[Sunscreen Aqua-Fresh] Beauty of Joseon Relief Sun '
                    'Aqua-Fresh Rice + B5 Spf50+ Pa++++ Rel